In [1]:
# Mount Drive e setup cartelle
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_PROJECT_PATH = "/content/drive/MyDrive/Progetto_EVWSD_ML"
EMBEDDINGS_DIR = os.path.join(DRIVE_PROJECT_PATH, "embeddings")
HF_CACHE_DIR = os.path.join(DRIVE_PROJECT_PATH, "data/hf_cache")

os.makedirs(EMBEDDINGS_DIR, exist_ok=True)
os.makedirs(HF_CACHE_DIR, exist_ok=True)

print("Cartelle pronte su Drive:", EMBEDDINGS_DIR)

Mounted at /content/drive
Cartelle pronte su Drive: /content/drive/MyDrive/Progetto_EVWSD_ML/embeddings


In [2]:
!pip install -q sentence-transformers datasets pillow torch

In [3]:
# Download ed estrazione imgs.zip
from huggingface_hub import hf_hub_download
import zipfile, os

IMAGES_DIR = os.path.join(DRIVE_PROJECT_PATH, "data/images")

if not os.path.exists(IMAGES_DIR) or len(os.listdir(IMAGES_DIR)) == 0:
    print("Download imgs.zip da HuggingFace...")
    zip_path = hf_hub_download(
        repo_id="swap-uniba/EVWSD-ITA",
        filename="imgs.zip",
        repo_type="dataset",
        cache_dir=HF_CACHE_DIR
    )
    print(f"Zip scaricato in: {zip_path}")
    print("Estrazione in corso...")
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall(IMAGES_DIR)
    print("Estrazione completata.")
else:
    print("Immagini già presenti, skip download.")

# ispezione: vedi i primi file estratti e l'estensione reale
all_files = []
for root, dirs, files in os.walk(IMAGES_DIR):
    for f in files:
        all_files.append(os.path.join(root, f))

print(f"\nTotale file immagine: {len(all_files)}")
print("Esempi:", all_files[:5])
print("Estensioni:", set(f.split(".")[-1] for f in all_files))

Immagini già presenti, skip download.

Totale file immagine: 13792
Esempi: ['/content/drive/MyDrive/Progetto_EVWSD_ML/data/images/imgs/F27/bn:00067892n.png', '/content/drive/MyDrive/Progetto_EVWSD_ML/data/images/imgs/F27/bn:00050746n.png', '/content/drive/MyDrive/Progetto_EVWSD_ML/data/images/imgs/F27/bn:00020403n.png', '/content/drive/MyDrive/Progetto_EVWSD_ML/data/images/imgs/F27/bn:00017073n.png', '/content/drive/MyDrive/Progetto_EVWSD_ML/data/images/imgs/F27/bn:00017067n.png']
Estensioni: {'png'}


In [4]:
#  Caricamento dataset da HuggingFace
from datasets import load_dataset

# dataset di training rilasciato per EVWSD-ITA
dataset = load_dataset(
    "swap-uniba/EVWSD-ITA",
    cache_dir=HF_CACHE_DIR
)

print(dataset)
# ispeziona la prima istanza per capire la struttura
print(dataset["train"][0])

README.md:   0%|          | 0.00/2.83k [00:00<?, ?B/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'hyp_id', 'gloss', 'lemma', 'hyp_lemma', 'bns', 'is_co_hyp', 'images', 'all_lemmas', 'all_glosses', 'img'],
        num_rows: 10000
    })
})
{'id': 'bn:00022412n', 'hyp_id': 'bn:00017670n', 'gloss': 'Atto del cuocere', 'lemma': 'cucina', 'hyp_lemma': ['cambiamento di stato'], 'bns': ['bn:00018237n', 'bn:00014468n', 'bn:00037474n', 'bn:00022423n', 'bn:00024323n', 'bn:00049248n'], 'is_co_hyp': [True, False, False, False, False, False], 'images': ['F14/bn:00018237n', 'F0/bn:00014468n', 'F0/bn:00037474n', 'F24/bn:00022423n', 'F26/bn:00024323n', 'F0/bn:00049248n'], 'all_lemmas': [['masticazione', 'masticare'], ['nave cambusa', 'cucina', 'cambusa'], ['culinaria', 'cucina', 'gastronomiche', 'gastronomo', 'gastronomia', 'gastronomica', 'arte culinaria', 'gastronomico'], ['cottura', 'cucina', 'stufa a legna', 'cucina a gas', 'cucina fornello', 'stufa cuoco', 'fornello', 'piano cottura'], ['culinaria', 'cottura', 'cucina', 'cucinare', 

In [5]:
import torch
from transformers import CLIPProcessor, CLIPModel

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

# vision encoder: CLIP base (stesso backbone del modello multilingue)
clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device)
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
clip_model.eval()
print("Modello CLIP caricato.")

Device: cuda


config.json:   0%|          | 0.00/4.19k [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/605M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/605M [00:00<?, ?B/s]

...

vocab.json:   0%|          | 0.00/862k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/525k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.22M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

Modello CLIP caricato.


In [6]:
from PIL import Image

IMG_EXT = ".png"
# lo zip ha estratto una sottocartella imgs/ dentro IMAGES_DIR
IMAGES_ROOT = os.path.join(IMAGES_DIR, "imgs")

def load_image(img_path_str):
    """
    Converte un identificatore tipo 'F22/bn:00022412n'
    nel path assoluto su disco e restituisce un PIL Image.
    """
    full_path = os.path.join(IMAGES_ROOT, img_path_str + IMG_EXT)
    if not os.path.exists(full_path):
        return None
    return Image.open(full_path).convert("RGB")

# --- verifica rapida prima di partire con i 10k ---
test = load_image("F22/bn:00022412n")
print("Test immagine target row[0]:", test)  # deve stampare <PIL.Image.Image ...>

Test immagine target row[0]: <PIL.Image.Image image mode=RGB size=336x336 at 0x7FD0B4159FA0>


In [7]:
from tqdm import tqdm

def encode_images(pil_images, model, processor, device, batch_size=32):
    """
    Prende una lista di PIL Image, le processa e restituisce
    un tensore di embedding normalizzati L2 di shape (N, 512).
    """
    all_embs = []
    for start in range(0, len(pil_images), batch_size):
        batch = pil_images[start:start+batch_size]
        inputs = processor(images=batch, return_tensors="pt", padding=True).to(device)
        with torch.no_grad():
            vision_out = model.vision_model(pixel_values=inputs["pixel_values"])
            emb = model.visual_projection(vision_out.pooler_output)
            emb = emb / emb.norm(dim=-1, keepdim=True)
        all_embs.append(emb.cpu())
    return torch.cat(all_embs, dim=0)

def compute_image_embeddings(dataset_split, model, processor, device,
                              batch_size=32, checkpoint_path=None):
    emb_target = {}
    emb_candidates = {}
    missing_target = 0

    for i in tqdm(range(len(dataset_split))):
        row = dataset_split[i]
        # usa l'indice di riga come chiave, non row["id"] che non è unico
        key = i

        img = load_image(row["img"])
        if img is not None:
            emb = encode_images([img], model, processor, device, batch_size)
            emb_target[key] = {
                "id": row["id"],
                "embedding": emb.squeeze(0)
            }
        else:
            missing_target += 1

        cands_emb = []
        for img_path in row["images"]:
            cand = load_image(img_path)
            if cand is not None:
                emb = encode_images([cand], model, processor, device, batch_size)
                cands_emb.append(emb.squeeze(0))
            else:
                cands_emb.append(None)
        emb_candidates[key] = {
            "id": row["id"],
            "embeddings": cands_emb
        }

        if checkpoint_path and i % 500 == 0 and i > 0:
            torch.save({"target": emb_target, "candidates": emb_candidates},
                       checkpoint_path)

    print(f"Target mancanti: {missing_target}/{len(dataset_split)}")
    return emb_target, emb_candidates

In [8]:
CHECKPOINT_PATH = os.path.join(EMBEDDINGS_DIR, "tensor_immagini_checkpoint.pt")
OUT_PATH        = os.path.join(EMBEDDINGS_DIR, "tensor_immagini.pt")

emb_target, emb_candidates = compute_image_embeddings(
    dataset["train"],
    clip_model,
    clip_processor,
    device,
    batch_size=32,
    checkpoint_path=CHECKPOINT_PATH
)

torch.save({"target": emb_target, "candidates": emb_candidates}, OUT_PATH)
print("Salvato in:", OUT_PATH)
print("Istanze con embedding target:", len(emb_target))
print("Istanze con embedding candidates:", len(emb_candidates))

100%|██████████| 10000/10000 [2:04:05<00:00,  1.34it/s]


Target mancanti: 0/10000
Salvato in: /content/drive/MyDrive/Progetto_EVWSD_ML/embeddings/tensor_immagini.pt
Istanze con embedding target: 10000
Istanze con embedding candidates: 10000
